# 4. Ensembles de Arboles de Decision

# -10º Corrida. Vuelvo a parámetros ya probados pero exploro feature_fraction = 0.3 y valor alto de maxdepth.

## **Adaptado para cumplir con lo solicitado en la clase 3:**
### - Considerar 32 árboles
### - Considerar cp=-1
### - Encontrar conjunto de parámetros eficientes
### - Subir a Kaggle resultados

## **Modificaciones que hago en el cuaderno para mejorar el proceso:**

### - Trabajo con una tabla intermedia donde creo todas las combinaciones de este cuaderno llamado CJ(). Así me evito tener que trabajar con iteraciones anidadas.

### - Agrego un archivo persistente que me ayuda a ir guardando lo realizado experimento por experimento. Así, en caso de colgarse Colab en medio de un proceso, lo reinicio y paso de largo por experimentos ya terminados.

### - Como veo en la planilla compartida que se pide la duración del tiempo de corrida, implemento una especie de cronómetro: guardo tiempo de inicio, tiempo de finalización, calculo la duración y la guardo en una columna que agregué en el archivo de resultados.




Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
# mantengo mi semilla primigenia personal en todos los experimentos
PARAM$semilla_primigenia <- 700001

# parámetro constante en todos los experimentos
PARAM$rpart$cp <- -1

# trabajaremos generando 32 árboles
PARAM$num_trees_max <- 32

# parámetros a explorar
PARAM$feature_fraction <- c(0.3, 0.4)

PARAM$rpart$minsplit <- c(300)
PARAM$rpart$minbucket <- c(100)
PARAM$rpart$maxdepth <- c(12, 20)



In [ ]:
# generamos todas las combinaciones de hiperparámetros
tb_grid <- CJ(
  feature_fraction = PARAM$feature_fraction,
  minsplit = PARAM$rpart$minsplit,
  minbucket = PARAM$rpart$minbucket,
  maxdepth = PARAM$rpart$maxdepth
)

# identificador de cada configuración
tb_grid[, id_experimento := .I]
tb_grid


In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp4020_10"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [ ]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [ ]:
# archivo persistente de control de experimentos
archivo_resultados <- "/content/buckets/b1/exp/resultados_experimentos.csv"

# si ya existe, lo leo
if (file.exists(archivo_resultados)) {

  tb_resultados <- fread(archivo_resultados)

  if (!"duracion_minutos" %in% names(tb_resultados)) {
    tb_resultados[, duracion_minutos := NA_real_]
    fwrite(tb_resultados, archivo_resultados)
    }

  message("Se encontraron ", nrow(tb_resultados)," experimentos ya realizados.")

} else {

  # si no existe, creo la estructura vacia
  tb_resultados <- data.table(
    id_experimento = integer(),
    feature_fraction = numeric(),
    minsplit = integer(),
    minbucket = integer(),
    maxdepth = integer(),
    cp = numeric(),
    arboles = integer(),
    archivo = character(),
    estado = character(),
    ganancia = numeric(),
    fecha_submission = character(),
    duracion_minutos = numeric()
  )

  fwrite(tb_resultados, archivo_resultados)

  message(
    "Se creo el archivo de control: ",
    archivo_resultados
  )
}

In [ ]:
Sys.setenv(TZ = "America/Argentina/Buenos_Aires")

In [ ]:
format(Sys.time(), "%Y-%m-%d %H:%M:%S")

for (i in seq_len(nrow(tb_grid))) {

  # parametros para esta configuracion
  feature_fraction_actual <- tb_grid[i, feature_fraction]
  minsplit_actual <- tb_grid[i, minsplit]
  minbucket_actual <- tb_grid[i, minbucket]
  maxdepth_actual <- tb_grid[i, maxdepth]

  #------- verifico si el experimento ya existe -----------
  registro_previo <- tb_resultados[id_experimento == i]

  if (nrow(registro_previo) > 0 && registro_previo$estado == "evaluado") {
    message("Experimento ", i, " ya realizado. Se omite.")
    next
  }
  #----------Fin de la verificación -----------------------

  cat(
    "\n=========================================\n",
    "Experimento:", i, "de", nrow(tb_grid), "\n",
    "feature_fraction:", feature_fraction_actual, "\n",
    "minsplit:", minsplit_actual, "\n",
    "minbucket:", minbucket_actual, "\n",
    "maxdepth:", maxdepth_actual, "\n",
    "\n=========================================\n",
    sep = ""
  )

  #----------- Si el experimento no existe calculo -------
  if (nrow(registro_previo) == 0) {

    # Inicio contador de tiempo
    tiempo_inicio <- Sys.time()

    # Asigno valor a la semilla
    set.seed(PARAM$semilla_primigenia)

    # Reinicio la tabla de predicciones
    tb_prediccion <- dfuture[, list(numero_de_cliente)]
    tb_prediccion[, prob_acumulada := 0]

    # Parámetros para esta configuración
    control_actual <- list(
      cp = PARAM$rpart$cp,
      minsplit = minsplit_actual,
      minbucket = minbucket_actual,
      maxdepth = maxdepth_actual
    )

    # Inicio el trabajo con los 32 árboles
    for (arbolito in seq_len(PARAM$num_trees_max)) {

      message(
        "Experimento ", i,
        " - arbol ", arbolito,
        " de ", PARAM$num_trees_max
      )

      qty_campos_a_utilizar <- as.integer(
        length(campos_buenos) * feature_fraction_actual
      )

      # elijo los campos al azar
      campos_random <- sample(
        campos_buenos,
        qty_campos_a_utilizar
      )

      # paso de un vector a un string con los elementos
      # separados por un signo de "+"
      # este hace falta para la formula
      campos_random <- paste(
        campos_random,
        collapse = " + "
      )

      # armo la formula para rpart
      formulita <- paste0(
        "clase_ternaria ~ ",
        campos_random
      )

      # genero el arbol de decision
      modelo <- rpart(
        formulita,
        data = dtrain,
        xval = 0,
        control = control_actual
      )

      # aplico el modelo a los datos que no tienen clase
      prediccion <- predict(
        modelo,
        dfuture,
        type = "prob"
      )

      # acumulo la probabilidad de BAJA+2
      tb_prediccion[
        ,
        prob_acumulada :=
          prob_acumulada + prediccion[, "BAJA+2"]
      ]
    }

    # Finalizo la contabilidad de tiempo
    tiempo_fin <- Sys.time()

    duracion_minutos <- as.numeric(
      difftime(
        tiempo_fin,
        tiempo_inicio,
        units = "mins"
      )
    )

    message(
      "Duracion de los 32 arboles: ",
      round(duracion_minutos, 2),
      " minutos"
    )

    # al terminar los 32 árboles
    umbral_corte <- (1 / 40) * PARAM$num_trees_max

    tb_prediccion[
      ,
      Predicted := as.numeric(
        prob_acumulada > umbral_corte
      )
    ]

    # nombre del archivo que generaremos
    nombre_archivo <- paste0(
      "KA420_",
      sprintf("%03d", i), # para que tenga ceros adelante
      "_ff", feature_fraction_actual,
      "_ms", minsplit_actual,
      "_mb", minbucket_actual,
      "_md", maxdepth_actual,
      ".csv"
    )

    archivo_kaggle <- paste0(
      "/content/buckets/b1/exp/",
      nombre_archivo
    )

    # grabo el archivo
    fwrite(
      tb_prediccion[, list(numero_de_cliente, Predicted)],
      file = archivo_kaggle,
      sep = ","
    )

    #-------------Registro del experimento terminado--------------
    # registro el experimento terminado
    nueva_fila <- data.table(
      id_experimento = i,
      feature_fraction = feature_fraction_actual,
      minsplit = minsplit_actual,
      minbucket = minbucket_actual,
      maxdepth = maxdepth_actual,
      cp = PARAM$rpart$cp,
      arboles = PARAM$num_trees_max,
      archivo = archivo_kaggle,
      estado = "generado",
      ganancia = NA_real_,
      fecha_submission = NA_character_,
      duracion_minutos = duracion_minutos
    )

    tb_resultados <- rbind(
      tb_resultados,
      nueva_fila,
      fill = TRUE
    )

    # guardo inmediatamente el estado actualizado
    fwrite(
      tb_resultados,
      archivo_resultados
    )

    message(
      "Experimento ",
      i,
      " generado y registrado."
    )
    #----------Fin del registro terminado -------------

  } else {

    #----------Ya estaba generado o enviado-------------
    #----------No vuelvo a calcular los 32 árboles -----

    archivo_kaggle <- registro_previo$archivo

    nombre_archivo <- basename(archivo_kaggle)

    message(
      "Experimento ",
      i,
      " ya calculado. No se recalculará."
    )
  }

  #------------ Consulto nuevamente el estado actual
  estado_actual <- tb_resultados[
    id_experimento == i,
    estado
  ]

  #----------------------------------------------------
  # subida a Kaggle
  #----------------------------------------------------
  if (estado_actual == "generado") {

    comando <- "kaggle competitions submit"
    competencia <- "-c utn-2026-inicial"
    arch <- paste("-f", archivo_kaggle)

    mensaje <- paste0(
      "-m 'exp=", i,
      " feature_fraction=", feature_fraction_actual,
      " cp=", PARAM$rpart$cp,
      " minsplit=", minsplit_actual,
      " minbucket=", minbucket_actual,
      " maxdepth=", maxdepth_actual,
      "'"
    )

    linea <- paste(
      comando,
      competencia,
      arch,
      mensaje
    )

    #------------VERIFICACION DE LINEA DE COMANDO
    cat("\nComando Kaggle:\n")
    cat(linea)
    cat("\n\n")
    #------------VERIFICACION DE LINEA DE COMANDO

    salida <- system(
      linea,
      intern = TRUE
    )

    cat(
      paste(salida, collapse = "\n"),
      "\n"
    )

    # Si llego hasta acá marco como enviado
    tb_resultados[
      id_experimento == i,
      estado := "enviado"
    ]

    fwrite(
      tb_resultados,
      archivo_resultados
    )

    message(
      "Experimento ",
      i,
      " enviado a Kaggle"
    )
  }

  # Espero a que Kaggle publique el resultado
  evaluado <- FALSE

  for (intento in 1:24) {

    Sys.sleep(5)  # Espero 5 segundos antes de preguntar

    salida_submissions <- system(
      "kaggle competitions submissions utn-2026-inicial -v -q",
      intern = TRUE
    )

    # Busco especificamente nuestro archivo
    fila_archivo <- salida_submissions[
      startsWith(
        salida_submissions,
        paste0(nombre_archivo, ",")
      )
    ]

    # Si Kaggle ya devolvio una fila para nuestro archivo
    if (length(fila_archivo) > 0) {

      # separo directamente los campos de la fila devuelta por Kaggle
      campos_kaggle <- strsplit(
        fila_archivo[1],
        ",",
        fixed = TRUE
      )[[1]]

      # proteccion ante una respuesta incompleta de Kaggle
      if (length(campos_kaggle) < 5) {
        message(
          "Respuesta inesperada de Kaggle. ",
          "Se intentara nuevamente."
        )
        next
      }

      # extraigo los campos que necesito
      fecha_kaggle <- campos_kaggle[2]
      estado_kaggle <- campos_kaggle[4]

      score_kaggle <- suppressWarnings(
        as.numeric(campos_kaggle[5])
      )

      # verifico si Kaggle ya termino de evaluarlo
      if (
        estado_kaggle == "SubmissionStatus.COMPLETE" &&
        !is.na(score_kaggle)
      ) {

        ganancia_kaggle <- score_kaggle * 1000000

        # actualizo la tabla de resultados
        tb_resultados[
          id_experimento == i,
          `:=`(
            estado = "evaluado",
            ganancia = ganancia_kaggle,
            fecha_submission = fecha_kaggle
          )
        ]

        # guardo inmediatamente en Drive
        fwrite(
          tb_resultados,
          archivo_resultados
        )

        message(
          "Experimento ",
          i,
          " evaluado. Ganancia = ",
          format(
            ganancia_kaggle,
            big.mark = ".",
            decimal.mark = ",",
            scientific = FALSE
          )
        )

        evaluado <- TRUE
        break
      }
    }

    message(
      "Esperando evaluacion de Kaggle... ",
      "intento ",
      intento,
      " de 24."
    )
  }

  #------------------------------------------------------------
  # si despues de 2 minutos Kaggle no respondio
  # no pierdo nada: queda registrado como ENVIADO
  #------------------------------------------------------------
  if (!evaluado) {
    message(
      "Kaggle todavia no devolvio el score del experimento ",
      i,
      ". Queda registrado como enviado."
    )
  }
}

format(Sys.time(), "%Y-%m-%d %H:%M:%S")